# 00. Запуск ETL

Запускает существующий ETL на всех строках `HI-Small_Trans.csv`. Каждый запуск создаёт отдельную папку и публикует `artifacts/runs/active.json` только после успешной записи.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").exists() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Запустите notebook из корня проекта или notebooks")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from aml.cli.etl import run_etl
from aml.data.read import load_dataset_splits
import pyarrow.parquet as pq
from time import perf_counter


In [2]:
SOURCE = PROJECT_ROOT / "data" / "raw" / "HI-Small_Trans.csv"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "runs"
SMOKE = False
if not SOURCE.is_file():
    raise FileNotFoundError(SOURCE)
print("Source:", SOURCE)
print("Size, MiB:", round(SOURCE.stat().st_size / 1024**2, 1))
print("Full dataset:", not SMOKE)


Source: /Users/artem/Documents/AML/AML/data/raw/HI-Small_Trans.csv
Size, MiB: 453.6
Full dataset: True


## Extract → validate → transform → load

Создает временные сплиты: train, val, threshold, test. При этом исторические признаки используют только события с более ранним timestamp

In [3]:
started = perf_counter()
RUN_DIR = run_etl(SOURCE, OUTPUT_DIR, smoke=SMOKE)
print("Completed run:", RUN_DIR)
print("Elapsed, minutes:", round((perf_counter() - started) / 60, 2))


Completed run: /Users/artem/Documents/AML/AML/artifacts/runs/20261011T162644Z_a8876ea20844
Elapsed, minutes: 1.01


## Проверка сохранённых файлов

Проверяем количество строк по Parquet metadata, не загружая все splits повторно в RAM.

In [4]:
manifest = json.loads((RUN_DIR / "data_manifest.json").read_text())
assert (RUN_DIR / "SUCCESS").is_file()
assert Path(manifest["source"]["path"]) == SOURCE.resolve()
configuration = json.loads((RUN_DIR / "configuration.json").read_text())
assert configuration["smoke"] is False
rows = []
for name in ("train", "val", "threshold", "test"):
    count = pq.ParquetFile(RUN_DIR / "data" / f"{name}.parquet").metadata.num_rows
    time_count = pq.ParquetFile(RUN_DIR / "data" / f"{name}_time.parquet").metadata.num_rows
    assert count == time_count > 0
    rows.append({"split": name, "rows": count, "timestamp_rows": time_count})
assert sum(row["rows"] for row in rows) == manifest["source"]["extracted_rows"]
display(pd.DataFrame(rows).set_index("split"))
display(pd.Series(json.loads((RUN_DIR / "quality_report.json").read_text())))
print("Next: 01_eda.ipynb, then 02_feature_engineering.ipynb")


,rows,timestamp_rows
split,,
train,3554649,3554649
val,507814,507814
threshold,253779,253779
test,762103,762103


rows                                                          5078345
columns                                                            11
missing_values      {'Timestamp': 0, 'From Bank': 0, 'Account': 0,...
duplicate_rows                                                      9
negative_amounts                                                    0
negatives                                                     5073168
positives                                                        5177
positive_rate                                                0.001019
min_timestamp                                     2022-09-01T00:00:00
max_timestamp                                     2022-09-18T16:18:00
dtype: object

Next: 01_eda.ipynb, then 02_feature_engineering.ipynb
